# ROS01 · ROS 2 nodes, topics and discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS01_ros_2_nodes_topics_and_discovery.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 60 min

**Skill Hub concepts:** ROS 2 nodes and topics; DDS discovery and domain ID

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS01")

## Map: the ROS 2 graph

```
  [joint1_sensor] ──/joint1/temperature (std_msgs/Float32, 20 Hz)──> [joint1_monitor] ──/joint1/overheat (Bool)──>
       node                       topic = name + type                    node
```

| Idea | ROS 2 meaning |
|---|---|
| Node | one process-level unit of computation with a name; owns publishers, subscriptions, timers, services |
| Topic | named, typed, many-to-many stream; publisher and subscriber match on **name, type and QoS** |
| Discovery | no master: every participant announces itself over DDS (UDP multicast), peers match automatically |
| Domain ID | `ROS_DOMAIN_ID` (0..101 safe range) splits DDS into isolated networks; different IDs never see each other |

DDS discovery port for a domain: $\text{port} = 7400 + 250 \times \text{domain\_id}$.

| CLI | Use |
|---|---|
| `ros2 node list` / `ros2 node info /name` | who is alive, what they publish |
| `ros2 topic list -t` / `ros2 topic info -v /t` | topics with types / endpoints with QoS |
| `ros2 topic echo /t` / `ros2 topic hz /t` | contents / rate |
| `ros2 interface show std_msgs/msg/Float32` | message definition |

In Colab you write each node to a file (`%%writefile`) and start it in the background with `ros.start(...)`,
exactly like opening a second terminal. The first ROS lab of a session installs ROS 2 Jazzy (3 to 5 minutes).

In [ ]:
from roboprep import ros
ros.ros("ros2 interface show std_msgs/msg/Float32")

## Task 1: a sensor node
Node name `joint1_sensor`; publish `std_msgs/msg/Float32` on `/joint1/temperature` at 20 Hz with value
$40 + 5\sin(t)$ ($t$ in seconds since start).

In [ ]:
%%writefile sensor_node.py
import math
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from std_msgs.msg import Float32


class Sensor(Node):
    def __init__(self):
        super().__init__("TODO")
        # TODO: publisher and a 20 Hz timer

    def tick(self):
        pass  # TODO


def main():
    rclpy.init()
    node = Sensor()
    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 2: a monitor node
Node name `joint1_monitor`; subscribe to `/joint1/temperature`; keep the last 10 values; publish `std_msgs/msg/Bool`
on `/joint1/overheat` after every message: `True` when the average of the last 10 is above 44.0.

In [ ]:
%%writefile monitor_node.py
from collections import deque
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from std_msgs.msg import Bool, Float32


class Monitor(Node):
    def __init__(self):
        super().__init__("joint1_monitor")
        pass  # TODO: deque(maxlen=10), Bool publisher, Float32 subscription

    def on_temp(self, msg):
        pass  # TODO


def main():
    rclpy.init()
    node = Monitor()
    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

Run both, then look at the graph the way you would from a terminal.

In [ ]:
import time
ros.start("python sensor_node.py", log="sensor.log")
ros.start("python monitor_node.py", log="monitor.log")
time.sleep(3)
ros.ros("ros2 node list --no-daemon --spin-time 2")
ros.ros("ros2 topic info -v /joint1/temperature | head -20")
ros.ros("timeout 3 ros2 topic hz /joint1/temperature", timeout=10)
ros.stop_all()

In [ ]:
lab.check("sensor")
lab.check("monitor")

In [ ]:
%%writefile sensor_node.cpp
// C++ (rclcpp) version of Task 1 (optional, same checker). Grade it with lab.check("sensor", "sensor_node.cpp"):
// the grader builds it with colcon as package rdl_sensor_node, executable sensor_node (depends on rclcpp, std_msgs).
#include <chrono>
#include <cmath>
#include <rclcpp/rclcpp.hpp>
#include <std_msgs/msg/float32.hpp>
using namespace std::chrono_literals;

class Sensor : public rclcpp::Node {
public:
    Sensor() : Node("TODO") {
        // TODO: Float32 publisher on /joint1/temperature and a 20 Hz wall timer publishing 40 + 5 sin(t)
    }
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<Sensor>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("sensor", "sensor_node.cpp")  # uncomment after writing the C++ cell above

In [ ]:
%%writefile monitor_node.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("monitor", "monitor_node.cpp"):
// the grader builds it with colcon as package rdl_monitor_node, executable monitor_node (depends on rclcpp, std_msgs).
#include <deque>
#include <rclcpp/rclcpp.hpp>
#include <std_msgs/msg/bool.hpp>
#include <std_msgs/msg/float32.hpp>

class Monitor : public rclcpp::Node {
public:
    Monitor() : Node("joint1_monitor") {
        // TODO: Bool publisher on /joint1/overheat, Float32 subscription on /joint1/temperature whose callback
        //       keeps the last 10 values and publishes average > 44.0
    }
private:
    std::deque<float> window_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<Monitor>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("monitor", "monitor_node.cpp")  # uncomment after writing the C++ cell above

Domain isolation: the same node on domain 8 is invisible from domain 7.

In [ ]:
ros.start("ROS_DOMAIN_ID=8 python sensor_node.py")
time.sleep(2)
ros.ros("echo 'domain 7 sees:'; ROS_DOMAIN_ID=7 ros2 node list --no-daemon --spin-time 2; echo 'domain 8 sees:'; ROS_DOMAIN_ID=8 ros2 node list --no-daemon --spin-time 2")
ros.stop_all()

## Checkpoint: Math
**M1 (Apply).** Period in milliseconds of a 20 Hz timer?

**M2 (Apply).** How many seconds of data does the 10-sample window cover at 20 Hz?

**M3 (Apply).** DDS discovery port for `ROS_DOMAIN_ID=7`?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Two teams test robots on the same Wi-Fi and see each other's topics. Cheapest fix?
- a) rename every topic
- b) give each team a different `ROS_DOMAIN_ID` (or limit discovery to localhost)
- c) switch to services

**K2 (Understand).** Explain how a subscriber in one process finds a publisher in another with no ROS master, and the
three things that must match before data flows.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros01.h`).
3. In `roboprep/checks/ros01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()